# Official fixture integrity checks on CPU
These checks use the actual installed organizer verifier on the clean base and
official public reference patch for rich_3067 and requests_6592. They do not run
Gemma, generate a patch or count as agent successes. Reference solutions remain
evaluator-only. Both cases are development cases excluded from future candidate
held-out studies. Original agent failures are not replaced by these diagnostics.

Attach the competition data and official wheelhouse. Use original 2026-07-01
environment, Accelerator None and Internet off. Run cells in order. Every check
exports a hash-verified evidence checkpoint; collect it while the next check runs.
No lost agent patches are reconstructed. Stop the worker after exporting results.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, base64, zipfile, io, subprocess, importlib, glob
from datetime import datetime, timezone
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_FIXTURE_CHECKS = True
DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')


In [ ]:
if IS_KAGGLE and RUN_FIXTURE_CHECKS:
    if not WHEELHOUSE_DIR.is_dir():
        raise RuntimeError('Attach the organizer wheelhouse before execution')
    for key, value in {'LITELLM_LOCAL_MODEL_COST_MAP':'True', 'TRANSFORMERS_NO_TF':'1',
        'VLLM_WORKER_MULTIPROC_METHOD':'spawn', 'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS':'1',
        'VLLM_ENGINE_READY_TIMEOUT_S':'1200', 'VLLM_NO_USAGE_STATS':'1',
        'OTEL_SDK_DISABLED':'true', 'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}.items():
        os.environ[key] = value
    # Organizer workaround for broken cutlass hooks, restricted to that package's .pth files.
    for pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
        for path in glob.glob(pattern): Path(path).unlink(missing_ok=True)
    wheel_dir = Path('/tmp/neural_swe_wheelhouse')
    wheel_dir.mkdir(exist_ok=True)
    for wheel in WHEELHOUSE_DIR.glob('*.whl'):
        if 'cutlass' in wheel.name.lower(): continue
        name = wheel.name.replace('cu128', '+cu128') if 'cu128' in wheel.name and '+' not in wheel.name else wheel.name
        link = wheel_dir / name
        if not link.exists(): link.symlink_to(wheel)
    wheels = sorted(str(p) for p in wheel_dir.glob('*.whl'))
    if not wheels: raise RuntimeError('Empty organizer wheelhouse')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                    '--no-deps', '--force-reinstall', *wheels], check=True)
    importlib.invalidate_caches()


In [ ]:
PLAN = json.loads('{"created_utc": "2026-10-03T15:04:25.237116+00:00", "scope": "public_development_not_held_out_or_leaderboard", "tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6", "selection": "minimum SHA256 of fixed salt and task ID within each declared repository; no reference or result screening", "tasks": [{"instance_id": "rich_3067", "repo": "Textualize/rich", "base_commit": "1ad6f73a4d1d3b40ac3f879feccfaad66412bd1d"}, {"instance_id": "requests_6592", "repo": "psf/requests", "base_commit": "0b4d494192de489701d3a2e32acef8fb5d3f042e"}], "arms": {"baseline": {"files": {"agent.yaml": "846e6e75c4845e790ba45f0b42884ac2d152b6c35368f546363bc3e6b4d666af", "prompts/system.md": "c7e1d29c42effae4c8b4f69c7cfaaceb13e8aa68c9f38ab492a20928bd1039b1", "configs/sampling.yaml": "6a66f5cd9328777d828e7111662e1d4923ca49e68161ebcde816184c3f2bc807"}}, "contract_cleanup_v2": {"files": {"agent.yaml": "846e6e75c4845e790ba45f0b42884ac2d152b6c35368f546363bc3e6b4d666af", "prompts/system.md": "cb96e0b69425dd395de0d2a5dcc21d5e9edf5eeee17bf40ac0cb2b7928c39481", "configs/sampling.yaml": "6a66f5cd9328777d828e7111662e1d4923ca49e68161ebcde816184c3f2bc807"}}}, "schedule": [{"task_id": "rich_3067", "arm": "baseline"}, {"task_id": "rich_3067", "arm": "contract_cleanup_v2"}, {"task_id": "requests_6592", "arm": "contract_cleanup_v2"}, {"task_id": "requests_6592", "arm": "baseline"}], "limits": {"timeout_seconds": 300, "max_time_minutes": 30, "max_tool_calls": 50, "max_turns": 100}, "same_model_and_backend": "official 31B QAT model revision 2 on L4 x4", "same_sampling": true, "sampling_seed_fixed": false, "repetitions_per_arm_per_case": 1, "reference_patches_exposed_to_agent": false, "exclude_from_future_candidate_held_out_studies": ["rich_3067", "requests_6592"], "existing_20261003_v1_study": "Preserved unchanged; this comparison does not evaluate or claim the locked memory-study effect."}')
if IS_KAGGLE and RUN_FIXTURE_CHECKS:
    RUN = Path('/kaggle/working') / ('neural_swe_fixture_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    (RUN / 'agent').mkdir()
    (RUN / 'plan.json').write_text(json.dumps(PLAN, indent=2) + '\n')
    print('Fixture output:', RUN)


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
from pathlib import Path
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifact_paths = sorted(p for p in run_root.rglob('*') if p.is_file()
                            and not p.is_symlink() and p.name != 'evidence_bundle.zip')
    manifest = []
    for artifact in artifact_paths:
        resolved = artifact.resolve()
        if not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact escaped its directory')
        manifest.append({'path': artifact.relative_to(run_root).as_posix(),
                         'bytes': artifact.stat().st_size,
                         'sha256': hashlib.sha256(artifact.read_bytes()).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for artifact in artifact_paths:
            archive.write(artifact, artifact.relative_to(run_root).as_posix())
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    (run_root / 'evidence_bundle.zip').write_bytes(bundle_bytes)
    report = {'sha256': hashlib.sha256(bundle_bytes).hexdigest(), 'bytes': len(bundle_bytes), 'file_count': len(manifest)}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
"""Check official public fixtures without model inference or agent success claims."""
import asyncio
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import inspect
import json
from pathlib import Path
import time


def classify_fixture(clean: dict, reference: dict) -> str:
    if clean['test_exit_code'] == -1 or reference['test_exit_code'] == -1:
        return 'grading_setup_failed'
    if reference['resolved'] is not True or reference['test_exit_code'] != 0:
        return 'official_reference_not_verified'
    if clean['test_exit_code'] == 0:
        return 'clean_base_already_passes'
    if clean['test_exit_code'] in (1, 2):
        return 'clean_base_nonpassing_reference_verified'
    return 'unclassified_nonpassing_clean_base'


def run_fixture_integrity(data_dir: Path, run_root: Path, plan: dict, checkpoint_exporter) -> dict:
    import torch
    from swegemma.evaluate import Evaluator
    from swegemma.config import EvalConfig
    from swegemma.models import load_tasks
    if torch.cuda.device_count() != 0:
        raise ValueError('Select Accelerator None: fixture verification uses CPU and no model')
    if hashlib.sha256((data_dir / 'tasks.jsonl').read_bytes()).hexdigest() != plan['tasks_sha256']:
        raise ValueError('Dataset differs from locked development source')
    if (run_root / 'fixture_summary.json').exists():
        raise FileExistsError('Preserve the previous fixture check; use a new run directory')
    module = inspect.getmodule(Evaluator)
    actual_tasks = {task.instance_id: task for task in load_tasks(data_dir / 'tasks.jsonl')}
    # The known JSON schema supplies patch; no guessed Python Task attribute is
    # required. Read it only for evaluator-side integrity checking, after the
    # original prompts/outcomes were locked. Never print reference patch text.
    raw_tasks = {row['instance_id']: row for row in map(json.loads,
                 (data_dir / 'tasks.jsonl').read_text().splitlines()) if row}
    report = {'scope': 'official_public_fixture_diagnostics_not_agent_trials',
              'created_utc': datetime.now(timezone.utc).isoformat(),
              'model_inference_executed': False, 'agent_generated_success_count': None,
              'official_leaderboard_score': None, 'gpu_count': torch.cuda.device_count(),
              'tasks_sha256': plan['tasks_sha256'],
              'guide_sha256': hashlib.sha256((data_dir / 'HARNESS_README.md').read_bytes()).hexdigest(),
              'packages': {name: importlib.metadata.version(name) for name in
                           ('swegemma', 'adk-submission', 'adk-eval-core')},
              'assigned_checks': 2 * len(plan['tasks']), 'observations': [], 'fixtures': [],
              'public_references_accessed_evaluator_only_after_agent_trials': True,
              'exclude_from_future_candidate_held_out_studies': [row['instance_id'] for row in plan['tasks']]}
    def save():
        (run_root / 'fixture_summary.json').write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    save()
    # Save the real installed API source for audit; it is diagnostic data.
    (run_root / 'verify_task_api.py').write_text(inspect.getsource(module.verify_task), encoding='utf-8')
    (run_root / 'resolve_snapshot_api.py').write_text(inspect.getsource(module.resolve_task_snapshot_paths), encoding='utf-8')

    async def evaluate_checks():
        for entry in plan['tasks']:
            task_id = entry['instance_id']
            task, raw_task = actual_tasks[task_id], raw_tasks[task_id]
            if raw_task['base_commit'] != entry['base_commit'] or raw_task['repo'] != entry['repo']:
                raise ValueError('Task identity differs from locked plan')
            snapshot, base_snapshot, task_patch = module.resolve_task_snapshot_paths(
                data_dir / 'snapshots', task_id, task.repo)
            results = {}
            for condition, patch in [('clean_base', ''), ('official_reference_patch', raw_task['patch'])]:
                destination = run_root / 'diagnostics' / task_id / condition
                destination.mkdir(parents=True, exist_ok=False)
                print(f'FIXTURE_CHECK_START {task_id} {condition}', flush=True)
                config = EvalConfig(tasks_path=data_dir / 'tasks.jsonl', snapshots_dir=data_dir / 'snapshots',
                    results_dir=destination, submission_dir=run_root / 'agent', models={}, sandbox='subprocess',
                    timeout_seconds=300, wheels_dir=data_dir / 'wheels',
                    graph_dir=str(data_dir / 'graphs'), embeddings_dir=str(data_dir / 'embeddings'))
                evaluator = Evaluator(config)
                result = await module.verify_task(evaluator.docker, config, task, snapshot,
                    base_snapshot_path=base_snapshot, patch_path=task_patch,
                    agent_patch=patch, start_time=time.perf_counter())
                details = result.model_dump(mode='json', exclude={'trace', 'agent_patch'})
                details.update(scope=report['scope'], condition=condition,
                               replayed_patch_sha256=hashlib.sha256(patch.encode()).hexdigest(),
                               reference_solution_text_not_exported=True)
                (destination / 'official_task_result.json').write_text(json.dumps(details, indent=2) + '\n', encoding='utf-8')
                (destination / 'test_output.txt').write_text(result.test_output or '', encoding='utf-8')
                observed = {'task_id': task_id, 'condition': condition, 'resolved': result.resolved,
                            'test_exit_code': result.test_exit_code,
                            'duration_seconds': result.duration_seconds,
                            'result_path': str(destination.relative_to(run_root) / 'official_task_result.json')}
                results[condition] = observed
                report['observations'].append(observed)
                save()
                print('FIXTURE_CHECK_RESULT ' + json.dumps(observed), flush=True)
                checkpoint_exporter(run_root)
            report['fixtures'].append({'task_id': task_id,
                'classification': classify_fixture(results['clean_base'], results['official_reference_patch'])})
            save()
        report['status'] = 'completed'
        save()
        checkpoint_exporter(run_root)
        return report
    try:
        with ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(evaluate_checks())).result()
    except Exception as error:
        report.update(status='infrastructure_exception', error_type=type(error).__name__, error=str(error))
        save()
        checkpoint_exporter(run_root)
        raise


In [ ]:
if IS_KAGGLE and RUN_FIXTURE_CHECKS:
    result = run_fixture_integrity(DATA_DIR, RUN, PLAN, export_run)
    print(json.dumps(result, indent=2))
else:
    print(json.dumps({'status':'local_definition_check_only', 'model_inference_executed':False,
                      'official_leaderboard_score':None}))
